# 7. SVM - Support Vector Machine

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')


In [2]:
# --- Load and preprocess churn dataset (Classification) ---
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df_churn = pd.read_csv('churn_df.csv')
# Dropping ID columns which have no predictive value
df_churn = df_churn.drop(columns=['RowNumber', 'CustomerId', 'Surname'], errors='ignore')

X_clf = df_churn.drop(columns=['Exited'])
y_clf = df_churn['Exited']

# Handling missing values
X_clf.fillna(X_clf.mean(numeric_only=True), inplace=True)
for col in X_clf.select_dtypes(include=['object']).columns:
    X_clf[col].fillna(X_clf[col].mode()[0], inplace=True)

# Categorical mapping using get_dummies
X_clf = pd.get_dummies(X_clf, drop_first=True)

# Train Test Split
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42)

# Scaling
scaler_clf = StandardScaler()
X_train_clf_scaled = scaler_clf.fit_transform(X_train_clf)
X_test_clf_scaled = scaler_clf.transform(X_test_clf)
print("Churn dataset ready for classification!")


Churn dataset ready for classification!


In [3]:
# --- Load and preprocess housing dataset (Regression) ---
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df_housing = pd.read_csv('housing.csv')
X_reg = df_housing.drop(columns=['median_house_value'])
y_reg = df_housing['median_house_value']

# Handling missing values
X_reg.fillna(X_reg.mean(numeric_only=True), inplace=True)
for col in X_reg.select_dtypes(include=['object']).columns:
    X_reg[col].fillna(X_reg[col].mode()[0], inplace=True)

# Categorical mapping using get_dummies
X_reg = pd.get_dummies(X_reg, drop_first=True)

# Train Test Split
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

# Scaling
scaler_reg = StandardScaler()
X_train_reg_scaled = scaler_reg.fit_transform(X_train_reg)
X_test_reg_scaled = scaler_reg.transform(X_test_reg)
print("Housing dataset ready for regression!")


Housing dataset ready for regression!


## SVM Classification (Churn Data)
Optimization: Added `class_weight='balanced'` for Class 1 recall.

In [4]:
from sklearn.svm import SVC

svm_clf = SVC(kernel='linear', class_weight='balanced', random_state=42)
svm_clf.fit(X_train_clf_scaled, y_train_clf)
svm_preds = svm_clf.predict(X_test_clf_scaled)

print("SVM Classification Accuracy:", accuracy_score(y_test_clf, svm_preds))
print(classification_report(y_test_clf, svm_preds))


SVM Classification Accuracy: 0.731
              precision    recall  f1-score   support

           0       0.92      0.73      0.81      1607
           1       0.40      0.73      0.52       393

    accuracy                           0.73      2000
   macro avg       0.66      0.73      0.67      2000
weighted avg       0.82      0.73      0.76      2000



## SVM Regression (Housing Data)
Optimization: Upgraded from linear kernel to highly potent `rbf` kernel (C=100) to cut MSE strictly and drastically on non-linear datasets!

In [5]:
from sklearn.svm import SVR

# SVM 'rbf' with normalized features is massively superior to linear format for geospatial data.
svm_reg = SVR(kernel='rbf', C=100, gamma='scale')
svm_reg.fit(X_train_reg_scaled, y_train_reg)
svm_reg_preds = svm_reg.predict(X_test_reg_scaled)

rmse = np.sqrt(mean_squared_error(y_test_reg, svm_reg_preds))
print(f"SVM Regression RMSE: ${rmse:,.2f}")
print("SVM Regression R2:", r2_score(y_test_reg, svm_reg_preds))


SVM Regression RMSE: $93,274.06
SVM Regression R2: 0.336081287502499
